# 15 — Subtype ResNet50 (pretrained + scratch)

Mirror of notebook 10 against the subtype labels. Same hyper-params so the comparison is direct.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
import yaml

from utils.seed import set_seed
from utils.models import build_resnet50
from utils.data_histology import LC25000Dataset, get_transforms
from utils.training import train_dl_cv, save_fold_results
from utils.metrics import aggregate_folds
from utils.stats import format_results_table

with open('../configs/histology_subtype.yaml') as f:
    cfg = yaml.safe_load(f)
set_seed(cfg['seed'])

cache_dir = Path('..') / cfg['data']['cache_dir']
labels_df = pd.read_csv(cache_dir / 'labels.csv')
y = labels_df['label_binary'].to_numpy().astype(int)
print('samples:', len(labels_df), 'pos rate (aca):', y.mean())

In [ ]:
train_tf, eval_tf = get_transforms(cfg['data']['image_size'])
dataset = LC25000Dataset(labels_df, transform=eval_tf, label_col='label_binary')

results_dir = Path('..') / cfg['paths']['results']
results_dir.mkdir(parents=True, exist_ok=True)

summary = {}

for tag, pretrained in [('pretrained', True), ('scratch', False)]:
    print(f'\n=== ResNet50 ({tag}) ===')

    def model_fn(_p=pretrained):
        return build_resnet50(pretrained=_p, num_classes=1)

    folds = train_dl_cv(
        dataset=dataset,
        model_fn=model_fn,
        labels=y,
        n_splits=cfg['cv']['n_splits'],
        seed=cfg['seed'],
        epochs=cfg['dl']['epochs'],
        batch_size=cfg['dl']['batch_size'],
        lr=cfg['dl']['lr'],
        weight_decay=cfg['dl']['weight_decay'],
        patience=cfg['dl']['patience'],
        num_workers=cfg['dl']['num_workers'],
        device='auto',
        augment_train=train_tf,
        augment_eval=eval_tf,
        verbose=True,
    )
    save_fold_results(folds, results_dir / f'subtype_ResNet50_{tag}.json')
    summary[f'ResNet50_{tag}'] = aggregate_folds([f.metrics_calibrated for f in folds])

format_results_table(summary)